<a href="https://colab.research.google.com/github/Chanaporn042/GE338-Lab-4/blob/main/Lab_4__Geographic_Modeling_6606614672.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.7 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import geemap.colormaps as cm

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-chanaporn040')  # ← ใส่ GEE Project ID ของคุณ

print('✅ GEE initialized successfully!')


✅ GEE initialized successfully!


In [ ]:
# ดึง boundary จังหวัดอุตรดิตถ์จาก FAO GAUL Level 2
thailand = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

uttaradit = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand')) \
    .filter(ee.Filter.eq('ADM1_NAME', 'Uttaradit'))

# แสดง boundary บนแผนที่
Map = geemap.Map(center=[17.6200, 100.0993], zoom=9)
Map.addLayer(uttaradit, {'color': '#2563eb', 'fillColor': '2563eb22'}, 'Uttaradit Boundary')
Map.addLayerControl()
Map

Map(center=[17.62, 100.0993], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

In [ ]:
# SRTM DEM 30m resolution
dem = ee.Image('USGS/SRTMGL1_003').clip(uttaradit)

# คำนวณ Slope (หน่วย: องศา)
slope = ee.Terrain.slope(dem)

# คำนวณ TWI = ln(A / tan(β))
# A = Specific Catchment Area (approximated), β = slope in radians
slope_rad = slope.multiply(ee.Number(3.14159265).divide(180))  # แปลง degree → radian
flow_acc = ee.Image('MERIT/Hydro/v1_0_1').select('upa').clip(uttaradit)  # upstream area (km²)
twi = flow_acc.divide(1e6).add(0.001).log() \
    .subtract(slope_rad.tan().add(0.001).log()) \
    .rename('TWI')

print('✅ DEM, Slope, TWI ready')

✅ DEM, Slope, TWI ready


In [ ]:
# HAND จาก MERIT-Hydro dataset
# hnd = height above nearest drainage (เมตร)
hand = ee.Image('MERIT/Hydro/v1_0_1').select('hnd').clip(uttaradit).rename('HAND')

# ตรวจสอบ statistics
hand_stats = hand.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True),
    geometry=uttaradit.geometry(),
    scale=1000,
    maxPixels=1e9
)
print('HAND stats:', hand_stats.getInfo())

HAND stats: {'HAND_max': 529.7000122070312, 'HAND_mean': 51.46468722922985, 'HAND_min': 0}


In [ ]:
# Sentinel-2 SR ปี 2023 (dry season: ม.ค.–เม.ย. เพื่อลด cloud cover)
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(uttaradit) \
    .filterDate('2023-01-01', '2023-04-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .median() \
    .clip(uttaradit)

# NDVI = (NIR - RED) / (NIR + RED)
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

print('✅ NDVI computed from Sentinel-2 2023')

✅ NDVI computed from Sentinel-2 2023


In [ ]:
def normalize(image, band_name, geometry, scale=1000):
    """Min-Max Normalization: (x - min) / (max - min)"""
    stats = image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e9
    )
    min_val = ee.Number(stats.get(band_name + '_min'))
    max_val = ee.Number(stats.get(band_name + '_max'))
    return image.subtract(min_val).divide(max_val.subtract(min_val)).rename(band_name + '_norm')

aoi = uttaradit.geometry()

# Normalize แต่ละ factor
hand_norm  = normalize(hand, 'HAND', aoi)
slope_norm = normalize(slope, 'slope', aoi)
ndvi_norm  = normalize(ndvi, 'NDVI', aoi)
twi_norm   = normalize(twi, 'TWI', aoi)

# Invert: ค่าสูง = เสี่ยงต่ำ → ต้อง invert เพื่อให้ 1 = เสี่ยงสูง
hand_risk  = ee.Image(1).subtract(hand_norm).rename('HAND_risk')   # ต่ำ = น้ำท่วมง่าย
slope_risk = ee.Image(1).subtract(slope_norm).rename('Slope_risk') # ราบ = ระบายช้า
ndvi_risk  = ee.Image(1).subtract(ndvi_norm).rename('NDVI_risk')   # ไม่มีพืช = เสี่ยงสูง
twi_risk   = twi_norm.rename('TWI_risk')                            # สูง = ชื้นสูง

print('✅ All factors normalized and inverted correctly')


✅ All factors normalized and inverted correctly


In [ ]:
# Weights (ต้องรวมกันได้ 1.0)
w_hand  = 0.35
w_slope = 0.25
w_ndvi  = 0.20
w_twi   = 0.20

# Weighted Linear Combination
flood_risk = hand_risk.multiply(w_hand) \
    .add(slope_risk.multiply(w_slope)) \
    .add(ndvi_risk.multiply(w_ndvi)) \
    .add(twi_risk.multiply(w_twi)) \
    .rename('Flood_Risk')

# แบ่งระดับความเสี่ยง (Thresholds)
# อ้างอิง: Tehrany et al. (2015) แบ่ง 5 class โดยใช้ Natural Breaks
flood_class = flood_risk \
    .where(flood_risk.lt(0.2), 1) \
    .where(flood_risk.gte(0.2).And(flood_risk.lt(0.4)), 2) \
    .where(flood_risk.gte(0.4).And(flood_risk.lt(0.6)), 3) \
    .where(flood_risk.gte(0.6).And(flood_risk.lt(0.8)), 4) \
    .where(flood_risk.gte(0.8), 5) \
    .rename('Flood_Class')

print('✅ WLC complete. Classes: 1=Very Low, 2=Low, 3=Moderate, 4=High, 5=Very High')



✅ WLC complete. Classes: 1=Very Low, 2=Low, 3=Moderate, 4=High, 5=Very High


In [ ]:
# Color palette: เขียว (ต่ำ) → เหลือง → แดง (สูง)
risk_palette = ['#1a9850', '#91cf60', '#ffffbf', '#fc8d59', '#d73027']

Map2 = geemap.Map(center=[17.6200, 100.0993], zoom=9)

# Layer ต่างๆ
Map2.addLayer(flood_risk, {'min': 0, 'max': 1, 'palette': risk_palette}, 'Flood Risk (Continuous)')
Map2.addLayer(flood_class, {'min': 1, 'max': 5, 'palette': risk_palette}, 'Flood Risk Class (1–5)')
Map2.addLayer(uttaradit, {'color': '#000000', 'fillColor': '00000000'}, 'Uttaradit Boundary')

# Legend
legend_dict = {
    '1 - Very Low': '#1a9850',
    '2 - Low': '#91cf60',
    '3 - Moderate': '#ffffbf',
    '4 - High': '#fc8d59',
    '5 - Very High': '#d73027'
}
Map2.add_legend(title='Flood Risk Level', legend_dict=legend_dict)
Map2.addLayerControl()
Map2


Map(center=[17.62, 100.0993], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

In [ ]:
#ทดสอบการเปลี่ยนน้ำหนัก HAND ±20% แล้วดูว่า class map เปลี่ยนมากน้อยแค่ไหน

#เซลล์โค้ด <undefined>
## %% [code]
def run_model(w_hand, w_slope, w_ndvi, w_twi):
    """รัน WLC ด้วย weights ที่กำหนด"""
    total = w_hand + w_slope + w_ndvi + w_twi
    risk = hand_risk.multiply(w_hand/total) \
        .add(slope_risk.multiply(w_slope/total)) \
        .add(ndvi_risk.multiply(w_ndvi/total)) \
        .add(twi_risk.multiply(w_twi/total))
    return risk.where(risk.lt(0.2), 1) \
               .where(risk.gte(0.2).And(risk.lt(0.4)), 2) \
               .where(risk.gte(0.4).And(risk.lt(0.6)), 3) \
               .where(risk.gte(0.6).And(risk.lt(0.8)), 4) \
               .where(risk.gte(0.8), 5)

# Base model
base_class    = run_model(0.35, 0.25, 0.20, 0.20)
# HAND +20%
hand_up_class = run_model(0.42, 0.25, 0.20, 0.20)
# HAND -20%
hand_dn_class = run_model(0.28, 0.25, 0.20, 0.20)

# คำนวณ Disagreement Map (พื้นที่ที่ class เปลี่ยน)
diff_up = base_class.neq(hand_up_class).rename('Diff_HAND_plus20')
diff_dn = base_class.neq(hand_dn_class).rename('Diff_HAND_minus20')
uncertainty = diff_up.add(diff_dn).rename('Uncertainty')  # 0=stable, 1=unstable in one, 2=unstable in both

# คำนวณ % พื้นที่ที่เปลี่ยน
def pct_changed(diff_img):
    stats = diff_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi, scale=1000, maxPixels=1e9
    )
    return ee.Number(stats.values().get(0)).multiply(100)

print('% pixels changed (HAND +20%):', pct_changed(diff_up).getInfo(), '%')
print('% pixels changed (HAND -20%):', pct_changed(diff_dn).getInfo(), '%')


% pixels changed (HAND +20%): 9.678143455732927 %
% pixels changed (HAND -20%): 10.44082155534289 %


In [ ]:
# แสดง Uncertainty Map
Map3 = geemap.Map(center=[17.6200, 100.0993], zoom=9)
Map3.addLayer(uncertainty, {'min': 0, 'max': 2, 'palette': ['#2d6a4f', '#ffd166', '#ef233c']}, 'Uncertainty Map')
Map3.addLayer(uttaradit, {'color': '#000000', 'fillColor': '00000000'}, 'Boundary')

uncertainty_legend = {
    'Stable (robust)': '#2d6a4f',
    'Unstable in 1 scenario': '#ffd166',
    'Unstable in both scenarios': '#ef233c'
}
Map3.add_legend(title='Model Uncertainty (HAND ±20%)', legend_dict=uncertainty_legend)
Map3


Map(center=[17.62, 100.0993], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

In [ ]:
# แก้ใหม่ทั้ง block
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(uttaradit)

observed_flood = jrc.gt(10).rename('Observed_Flood')
predicted_high = flood_class.gte(4).rename('Predicted_High')

# ✅ วิธีที่ถูก: ใช้ updateMask เพื่อให้นับเฉพาะ pixel ที่มีข้อมูลทั้งคู่
valid_mask = jrc.mask().And(flood_class.mask())

observed = observed_flood.updateMask(valid_mask)
predicted = predicted_high.updateMask(valid_mask)

# สร้าง combined image แล้วนับ
tp_img = predicted.eq(1).And(observed.eq(1))
fp_img = predicted.eq(1).And(observed.eq(0))
fn_img = predicted.eq(0).And(observed.eq(1))
tn_img = predicted.eq(0).And(observed.eq(0))

def count_pixels(img):
    s = img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=300,       # ✅ ลด scale ลง — 1000m หยาบเกินไป
        maxPixels=1e9,
        bestEffort=True  # ✅ ป้องกัน timeout
    )
    return ee.Number(s.values().get(0))

tp = count_pixels(tp_img)
fp = count_pixels(fp_img)
fn = count_pixels(fn_img)
tn = count_pixels(tn_img)

tp_v, fp_v, fn_v, tn_v = tp.getInfo(), fp.getInfo(), fn.getInfo(), tn.getInfo()
total = tp_v + fp_v + fn_v + tn_v

print(f'Total valid pixels: {total}')
print(f'TP={tp_v}, FP={fp_v}, FN={fn_v}, TN={tn_v}')

if total > 0:
    precision = tp_v / (tp_v + fp_v + 1e-9)
    recall    = tp_v / (tp_v + fn_v + 1e-9)
    accuracy  = (tp_v + tn_v) / total
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    print(f'Precision: {precision:.3f}')
    print(f'Recall   : {recall:.3f}')
    print(f'Accuracy : {accuracy:.3f}')
    print(f'F1-Score : {f1:.3f}')
else:
    print('❌ ไม่มี pixel ที่มีข้อมูลทั้งสอง layer — ตรวจสอบ mask ของ JRC')

Total valid pixels: 9260.282352941176
TP=5017.066666666667, FP=4037.827450980392, FN=132.29411764705884, TN=73.09411764705882
Precision: 0.554
Recall   : 0.974
Accuracy : 0.550
F1-Score : 0.706


In [ ]:
confusion = (ee.Image(0)
    .where(tp_img, 1) # TP
    .where(fp_img, 2) # FP
    .where(fn_img, 3) # FN
    .where(tn_img, 4))

confusion_masked = confusion.selfMask()

Map4 = geemap.Map(center=[17.6200, 100.0993], zoom=9)
Map4.addLayer(confusion_masked, {
    'min': 1, 'max': 4,
    'palette': ['#2ecc71', '#e74c3c', '#f39c12', '#bdc3c7']
}, 'Validation Map')
Map4.addLayer(uttaradit, {'color': '#000000', 'fillColor': '00000000'}, 'Boundary')

val_legend = {
    'TP — Correctly predicted flood': '#2ecc71',
    'FP — Predicted flood, no actual flood': '#e74c3c',
    'FN — Missed actual flood': '#f39c12',
    'TN — Correctly predicted no flood': '#bdc3c7'
}
Map4.add_legend(title='Validation vs JRC Surface Water', legend_dict=val_legend)
Map4

Map(center=[17.62, 100.0993], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

In [ ]:
# ======================================
# Task 8: Export ผลลัพธ์ไป Google Drive
# ======================================

# 1. Export Flood Risk Class Map
export_task = ee.batch.Export.image.toDrive(
    image=flood_class.toByte(),
    description='Flood_Risk_Uttaradit',
    folder='Lab4',
    fileNamePrefix='flood_risk_uttaradit_5class',
    region=uttaradit.geometry().bounds(),
    scale=100,
    crs='EPSG:32647',  # UTM Zone 47N เหมาะกับภาคเหนือไทย
    maxPixels=1e9
)
export_task.start()
print("✅ Export Flood Risk Class started!")

# 2. Export Flood Risk Continuous (ค่า 0-1)
export_risk = ee.batch.Export.image.toDrive(
    image=flood_risk.toFloat(),
    description='Flood_Risk_Continuous_Uttaradit',
    folder='Lab4',
    fileNamePrefix='flood_risk_continuous',
    region=uttaradit.geometry().bounds(),
    scale=100,
    crs='EPSG:32647',
    maxPixels=1e9
)
export_risk.start()
print("✅ Export Flood Risk Continuous started!")

# 3. Export Uncertainty Map
export_unc = ee.batch.Export.image.toDrive(
    image=uncertainty.toByte(),
    description='Uncertainty_Uttaradit',
    folder='Lab4',
    fileNamePrefix='uncertainty_uttaradit',
    region=uttaradit.geometry().bounds(),
    scale=100,
    crs='EPSG:32647',
    maxPixels=1e9
)
export_unc.start()
print("✅ Export Uncertainty started!")

# ตรวจสอบสถานะ
print("\n📋 Task Status:")
print("Flood Class  :", export_task.status()['state'])
print("Flood Risk   :", export_risk.status()['state'])
print("Uncertainty  :", export_unc.status()['state'])

✅ Export Flood Risk Class started!
✅ Export Flood Risk Continuous started!
✅ Export Uncertainty started!

📋 Task Status:
Flood Class  : READY
Flood Risk   : READY
Uncertainty  : READY
